# IT Support Ticket Classification & Priority Prediction
### Real-World NLP Machine Learning Project

---

**Dataset:** Zenodo IT Support Tickets — 2,229 real human-classified tickets  
**Source:** https://zenodo.org/records/7648117  
**Tools:** Python · Scikit-learn · Pandas · Matplotlib · Seaborn · Streamlit

---

## What This Project Does

Reads raw IT support ticket text and automatically:
1. **Classifies** into category — Hardware / Software / Network / Access / Storage / Printing / Administrative Rights
2. **Predicts priority** — High / Medium / Low (engineered from keyword rules since dataset has no priority column)

## Table of Contents
1. Setup & Data Loading  
2. Exploratory Data Analysis  
3. Priority Engineering  
4. Text Preprocessing  
5. Feature Engineering (TF-IDF)  
6. Category Classification  
7. Priority Prediction  
8. Results & Insights  
9. Live Demo

---
## 1. Setup & Data Loading

**Before running this notebook:**  
Download from https://zenodo.org/records/7648117  
Place `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` inside the `data/` folder.

In [ ]:
import re, string, warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

COLORS     = ['#457b9d','#e63946','#f4a261','#2a9d8f','#8338ec','#06d6a0','#ffb703']
PRI_COLORS = {'High':'#e63946','Medium':'#f4a261','Low':'#2a9d8f'}
print('All libraries loaded!')

: 

In [ ]:
# Load the four Zenodo CSV files
X_train = pd.read_csv('data/X_train.csv')
X_test  = pd.read_csv('data/X_test.csv')
y_train = pd.read_csv('data/y_train.csv')
y_test  = pd.read_csv('data/y_test.csv')

# Check column names
print('X_train columns:', list(X_train.columns))
print('y_train columns:', list(y_train.columns))

In [ ]:
# Identify text and label columns automatically
text_col  = X_train.select_dtypes(include='object').columns[0]
label_col = y_train.columns[0]
print(f'Text column : {text_col}')
print(f'Label column: {label_col}')

# Merge into one DataFrame
df_train = pd.DataFrame({'text': X_train[text_col].values,
                          'category': y_train[label_col].values, 'split': 'train'})
df_test  = pd.DataFrame({'text': X_test[text_col].values,
                          'category': y_test[label_col].values,  'split': 'test'})
df = pd.concat([df_train, df_test], ignore_index=True)
df['category'] = df['category'].astype(str).str.strip().str.title()
df = df.dropna(subset=['text','category'])
df = df[df['text'].str.strip() != '']

print(f'\nTotal tickets: {len(df)}')
print(f'Categories   : {df["category"].nunique()}')
df.head(3)

---
## 2. Exploratory Data Analysis

In [ ]:
print('Category distribution:')
print(df['category'].value_counts())

fig, ax = plt.subplots(figsize=(10, 5))
cc = df['category'].value_counts()
ax.barh(cc.index, cc.values, color=COLORS[:len(cc)], alpha=0.88)
for i, v in enumerate(cc.values):
    ax.text(v + 1, i, str(v), va='center', fontsize=10)
ax.invert_yaxis()
ax.set_title('IT Support Tickets by Category (Zenodo Dataset)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Tickets')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

: 

In [ ]:
# Look at real ticket samples
print('=== SAMPLE TICKETS ===')
for cat in df['category'].unique()[:3]:
    sample = df[df['category'] == cat]['text'].iloc[0]
    print(f'\n[{cat}]')
    print(sample[:300])
    print('-'*60)

---
## 3. Priority Engineering

The Zenodo dataset does not include a priority column — only categories.  
This is common in real-world data. We solve it using **rule-based labelling**: assigning priority based on keyword signals in the ticket text.

This is a valid real-world technique used in production systems.

In [ ]:
HIGH_KEYWORDS = [
    r'\b(?:urgent|asap|immediately|critical|emergency|down|outage)\b',
    r'\b(?:crash|crashed|crashing|broken|not working|stopped working)\b',
    r'\b(?:production|server|database|network|infrastructure)\b',
    r'\b(?:security|breach|hack|compromised|virus|malware|ransomware)\b',
    r'\b(?:data loss|lost data|cannot access|locked out|blocked)\b',
    r'!{2,}',
]
LOW_KEYWORDS = [
    r'\b(?:question|wondering|how do|how can|where is|what is)\b',
    r'\b(?:request|suggestion|feature|improvement|enhancement)\b',
    r'\b(?:information|documentation|guide|tutorial|manual)\b',
    r'\b(?:when will|roadmap|future|plan|schedule)\b',
]

def assign_priority(text):
    text = str(text).lower()
    high = sum(1 for p in HIGH_KEYWORDS if re.search(p, text))
    low  = sum(1 for p in LOW_KEYWORDS  if re.search(p, text))
    if high >= 2 or (high == 1 and low == 0): return 'High'
    if low  >= 2 or (low  == 1 and high == 0): return 'Low'
    return 'Medium'

df['priority'] = df['text'].apply(assign_priority)

print('Priority distribution (engineered):')
print(df['priority'].value_counts())

fig, ax = plt.subplots(figsize=(6, 5))
pc = df['priority'].value_counts().reindex(['High','Medium','Low'])
ax.pie(pc.values, labels=pc.index, autopct='%1.1f%%',
       colors=[PRI_COLORS[p] for p in pc.index],
       startangle=140, wedgeprops=dict(edgecolor='white',linewidth=2))
ax.set_title('Engineered Priority Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Text Preprocessing

The Zenodo dataset is multilingual and already has PII replaced with tags like `[NAME]`, `[EMAIL]`. We clean all of this along with standard noise.

In [ ]:
STOPWORDS = {
    'i','me','my','we','our','you','your','he','him','his','she','her','it','its',
    'they','them','their','what','which','who','this','that','these','those','am',
    'is','are','was','were','be','been','being','have','has','had','do','does','did',
    'a','an','the','and','but','if','or','of','at','by','for','with','to','from',
    'in','out','on','off','up','down','no','not','will','just','should','can','may',
    'please','would','could','get','also','want','need','use','know','like','make',
    'hi','hello','dear','regards','thank','thanks','sincerely','kindly',
    # Portuguese
    'eu','tu','ele','ela','nos','um','uma','de','da','do','em','na','no','por','para',
    'que','se','mais','mas','ao','dos','das','como','foi','ser','ter',
    # German
    'ich','du','er','sie','es','wir','den','dem','der','die','das','ein','eine',
    'und','oder','aber','nicht','mit','auf','bei','von','zu','ist','hat','kann',
    # Spanish
    'yo','el','los','las','del','al','por','pero','muy','este','esta',
}

PII_TAGS = {'[name]','[email]','[phone]','[address]','[url]',
            '[date]','[number]','[id]','[ip]','[organization]','[location]'}

def clean_text(text):
    text = str(text).lower()
    for tag in PII_TAGS:           # remove PII placeholder tags
        text = text.replace(tag, ' ')
    text = re.sub(r'\[.*?\]', ' ', text)              # remaining brackets
    text = re.sub(r'http\S+|www\.\S+', ' ', text)    # URLs
    text = re.sub(r'\S+@\S+', ' ', text)             # emails
    text = re.sub(r'\d+', ' ', text)                 # numbers
    text = text.translate(
        str.maketrans(string.punctuation, ' '*len(string.punctuation)))
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) >= 2]
    return ' '.join(tokens)

# Test
sample = 'URGENT: [NAME] cannot access the server at [IP]. Please help immediately!!'
print('BEFORE:', sample)
print('AFTER :', clean_text(sample))

In [ ]:
df['clean_text']   = df['text'].apply(clean_text)
df['char_len']     = df['text'].str.len()
df['word_count']   = df['text'].str.split().str.len()
df['exclaim']      = df['text'].str.count(r'!')
df['question']     = df['text'].str.count(r'\?')
urgency_pat = r'\b(?:urgent|asap|critical|emergency|broken|crash|production|hack|down|outage)\b'
df['urgency_flag'] = df['text'].str.lower().str.contains(urgency_pat, regex=True).astype(int)

print('Features added. Sample cleaned text:')
print(df['clean_text'].iloc[0][:200])

---
## 5. Feature Engineering (TF-IDF)

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=10000,
                         sublinear_tf=True, min_df=2, max_df=0.92)
X_tfidf = tfidf.fit_transform(df['clean_text'])

META_COLS = ['char_len','word_count','exclaim','question','urgency_flag']
scaler  = MaxAbsScaler()
X_meta  = scaler.fit_transform(csr_matrix(df[META_COLS].values))
X       = hstack([X_tfidf, X_meta])
print(f'Feature matrix: {X.shape[0]} tickets x {X.shape[1]} features')

In [ ]:
# Top TF-IDF keywords per category
feature_names = np.array(tfidf.get_feature_names_out())
cats = df['category'].unique()
n    = len(cats)

fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1: axes = [axes]
for ax, cat, color in zip(axes, cats, COLORS):
    mask  = df['category'] == cat
    sub   = tfidf.transform(df.loc[mask,'clean_text'])
    mean  = np.asarray(sub.mean(axis=0)).flatten()
    top   = mean.argsort()[-12:][::-1]
    ax.barh(range(12), mean[top][::-1], color=color, alpha=0.85)
    ax.set_yticks(range(12))
    ax.set_yticklabels(feature_names[top][::-1], fontsize=8)
    ax.set_title(cat, fontsize=9, fontweight='bold', color=color)
    ax.set_xlabel('Mean TF-IDF', fontsize=8)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Top Keywords per Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Category Classification

In [ ]:
le_cat = LabelEncoder()
le_pri = LabelEncoder()
y_cat  = le_cat.fit_transform(df['category'])
y_pri  = le_pri.fit_transform(df['priority'])

X_tr, X_te, yc_tr, yc_te, yp_tr, yp_te = train_test_split(
    X, y_cat, y_pri, test_size=0.20, random_state=42, stratify=y_cat)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = [
    ('Logistic Regression', LogisticRegression(max_iter=1000, C=5)),
    ('Linear SVC',          LinearSVC(max_iter=2000, C=1.0)),
    ('Naive Bayes',         MultinomialNB(alpha=0.1)),
    ('Random Forest',       RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)),
]

print('Category Model Comparison:')
cat_results = []
for name, clf in models:
    sc = cross_val_score(clf, X_tr, yc_tr, cv=cv, scoring='accuracy')
    cat_results.append((name, sc.mean(), sc.std()))
    print(f'  {name:<25} {sc.mean():.4f} +/- {sc.std():.4f}')

In [ ]:
best_cat = LogisticRegression(max_iter=1000, C=5)
best_cat.fit(X_tr, yc_tr)
yc_pred = best_cat.predict(X_te)
cat_acc = accuracy_score(yc_te, yc_pred)

print(f'Category Test Accuracy: {cat_acc*100:.1f}%')
print()
print(classification_report(yc_te, yc_pred, target_names=le_cat.classes_))

cm = confusion_matrix(yc_te, yc_pred)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_cat.classes_, yticklabels=le_cat.classes_,
            linewidths=0.5, ax=ax)
ax.set_title(f'Category Confusion Matrix (Acc: {cat_acc*100:.1f}%)',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 7. Priority Prediction

In [ ]:
print('Priority Model Comparison:')
pri_results = []
for name, clf in models:
    sc = cross_val_score(clf, X_tr, yp_tr, cv=cv, scoring='accuracy')
    pri_results.append((name, sc.mean(), sc.std()))
    print(f'  {name:<25} {sc.mean():.4f} +/- {sc.std():.4f}')

best_pri = LogisticRegression(max_iter=1000, C=3)
best_pri.fit(X_tr, yp_tr)
yp_pred = best_pri.predict(X_te)
pri_acc = accuracy_score(yp_te, yp_pred)

print(f'\nPriority Test Accuracy: {pri_acc*100:.1f}%')
print()
print(classification_report(yp_te, yp_pred, target_names=le_pri.classes_))

cm2 = confusion_matrix(yp_te, yp_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le_pri.classes_, yticklabels=le_pri.classes_,
            linewidths=0.5, ax=ax)
ax.set_title(f'Priority Confusion Matrix (Acc: {pri_acc*100:.1f}%)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Results & Insights

In [ ]:
print('='*50)
print('         FINAL RESULTS')
print('='*50)
print(f'  Category Accuracy : {cat_acc*100:.1f}%')
print(f'  Priority Accuracy : {pri_acc*100:.1f}%  (rule-engineered labels)')
print(f'  Category Baseline : {100/len(le_cat.classes_):.1f}%  (random)')
print(f'  Priority Baseline : 33.3%  (random)')
print('='*50)
print()
print('Key insight: Priority accuracy is higher than baseline because')
print('our rule-based labels are consistent — the model learns the same')
print('keywords we used to create the labels. In production, human-labelled')
print('priority data would give more realistic (and lower) accuracy.')

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(tfidf,    'models/tfidf.pkl')
joblib.dump(scaler,   'models/scaler.pkl')
joblib.dump(best_cat, 'models/cat_model.pkl')
joblib.dump(best_pri, 'models/pri_model.pkl')
joblib.dump(le_cat,   'models/le_cat.pkl')
joblib.dump(le_pri,   'models/le_pri.pkl')
print('All models saved to models/')

---
## 9. Live Demo

In [ ]:
def predict_ticket(text):
    cleaned = clean_text(text)
    x_tfidf = tfidf.transform([cleaned])
    meta = pd.DataFrame([{
        'char_len': len(text), 'word_count': len(text.split()),
        'exclaim': text.count('!'), 'question': text.count('?'),
        'urgency_flag': int(bool(re.search(
            r'\b(?:urgent|asap|critical|emergency|broken|crash|production|hack|down|outage)\b',
            text.lower())))
    }])
    x_meta = scaler.transform(csr_matrix(meta[META_COLS].values))
    x = hstack([x_tfidf, x_meta])
    return {
        'category': le_cat.inverse_transform(best_cat.predict(x))[0],
        'priority': le_pri.inverse_transform(best_pri.predict(x))[0],
    }

META_COLS = ['char_len','word_count','exclaim','question','urgency_flag']

demo_tickets = [
    'URGENT: The entire network is down. No one in the office can connect to the internet.',
    'My computer is running very slowly. It takes 5 minutes to start up.',
    'I need access to the accounting software. Can you give me the credentials?',
    'The printer on floor 3 is not printing. Can someone check it?',
    'When will the new version of the ERP software be available?',
]

print(f'{"TICKET":<55} {"CATEGORY":<25} PRIORITY')
print('-'*90)
for t in demo_tickets:
    r = predict_ticket(t)
    print(f"{t[:53]:<55} {r['category']:<25} {r['priority']}")